# Introducción

México enfrenta una paradoja agrícola persistente: es el primer productor mundial de aguacate y uno de los mayores exportadores de limón, pero los pequeños y medianos productores que sostienen esa cadena rara vez se benefician del valor que generan. El origen de esta brecha no es únicamente logístico ni financiero — es **informacional**.

Cuando un productor de Michoacán o Veracruz decide cuándo y a quién vender su cosecha, lo hace casi siempre sin saber si el precio que le ofrecen ese día es bueno o malo respecto a lo que esa misma ruta ha pagado históricamente. Los intermediarios que compran en origen y revenden en mercados mayoristas como la Central de Abastos de la Ciudad de México sí tienen esa información. Esa asimetría determina quién captura el valor.

**Mercado Justo** es nuestra propuesta para cerrar esa brecha. El proyecto sistematiza datos oficiales de precios del SNIIM entre 2000 y 2026, y construye un pipeline que convierte esa historia de precios en recomendaciones concretas y accionables para el productor: ¿conviene vender hoy en esta ruta? ¿cuánto volumen destinar a cada mercado? ¿cuál es el precio esperado para los próximos meses?

El pipeline se compone de seis etapas encadenadas, donde la salida de cada una alimenta a la siguiente:

1. **Adquisición de Datos** (Web Scraping) — recolecta precios diarios directamente del portal del SNIIM mediante un motor interactivo de scraping.
2. **Análisis exploratorio** (PySpark) — caracteriza la distribución de precios, las rutas activas y la estacionalidad de Aguacate Hass y Limón con semilla a lo largo de una década.
3. **Pronóstico SARIMA** — estima el precio promedio esperado para los próximos 12 meses, capturando los ciclos anuales propios de cada cultivo.
4. **Random Forest** — aprende, a partir de criterios históricos por ruta, cuándo el precio actual de un origen–destino está en un momento favorable; lo hace sin *label leakage*, usando únicamente información observable al momento de la decisión.
5. **Algoritmo de Dijkstra** — determina la ruta logística de menor costo para conectar el estado productor con los principales mercados, ajustando el costo de flete según la probabilidad de obtener un precio favorable dictada por el modelo predictivo.
6. **Plan de marketing digital** — traduce las recomendaciones en alertas accionables para el productor vía WhatsApp y correo electrónico.

El procesamiento corre íntegramente sobre **PySpark**, lo que permite trabajar sobre el histórico completo sin muestreo ni pérdida de información.

***
**Datos**: SNIIM (2000–2026) | **Productos**: Aguacate Hass · Limón con semilla | **Motor**: Scraping + PySpark + SARIMA + Random Forest
***

# Adquisición de Datos – Web Scraping interactivo {#sec-scraping}

Obtener datos del **Sistema Nacional de Información e Integración de Mercados (SNIIM)** representa un desafío técnico, ya que el portal no ofrece una API pública ni formatos de descarga masiva directa. Para solucionar esto, se desarrolló un motor de automatización mediante **Web Scraping**.

Esta etapa es fundamental porque permite la actualización constante de la base de datos sin intervención manual, garantizando que las predicciones se basen en la realidad más reciente del mercado. Para su implementación, se utilizaron las siguientes librerías:

- **BeautifulSoup y Requests:** Permiten navegar la estructura HTML del portal del SNIIM y extraer las tablas de precios de forma precisa.
- **Pandas:** Facilita la limpieza inmediata y estructuración de los datos recolectados en un formato tabular.
- **Ipywidgets:** Proporciona una interfaz interactiva para filtrar productos y rangos de fechas antes de iniciar la extracción.

La automatización de este proceso no solo ahorra horas de trabajo manual, sino que elimina errores humanos en la captura de datos, permitiendo que el pipeline de **Mercado Justo** sea escalable y robusto.

Puedes consultar y ejecutar el proceso completo de extracción en el siguiente enlace:

[**Acceder al Web Scraping en Google Colab**](https://colab.research.google.com/drive/1Rmnhu06-Y-VS5UO5_4jQjl8RtYpdsHdS)

---

# Configuración {#sec-config}

## Entorno de ejecución

El proyecto fue desarrollado dentro de un entorno reproducible basado en **Docker**. Para ello se utilizó una imagen derivada de `jupyter/pyspark-notebook`, extendida mediante un **Dockerfile** para incorporar herramientas adicionales como **Quarto**, utilizado en la generación de este reporte.

Una vez creado el contenedor, su ejecución cotidiana se realiza mediante:

```bash
docker start mercado-justo
```

El entorno expone los siguientes servicios:

| Puerto | Servicio |
|---------|----------|
| `8888` | Jupyter Lab / Notebook |
| `4040` | Spark UI para monitoreo de procesos |

Asimismo, se configuró un volumen compartido entre el sistema anfitrión y el contenedor para garantizar la persistencia de notebooks, datos y reportes generados durante el desarrollo del proyecto.

Esta configuración permite reproducir de manera consistente el flujo completo de análisis, modelado y visualización empleado en Mercado Justo.

## Librerías y parámetros

El bloque de configuración centraliza todas las dependencias del proyecto y los parámetros
ajustables en un único lugar, facilitando la reproducibilidad y el mantenimiento del pipeline.

| Categoría | Librería / Parámetro | Rol en el proyecto |
|---|---|---|
| **Datos y transformación** | `PySpark` — `SparkSession`, `functions`, `types` | Motor de procesamiento distribuido sobre el histórico completo del SNIIM |
| **Machine Learning** | `RandomForestClassifier`, `VectorAssembler`, `Pipeline` | Clasificación de momentos favorables de venta por ruta |
| **Evaluación ML** | `MulticlassClassificationEvaluator`, `BinaryClassificationEvaluator` | Métricas de desempeño del modelo de clasificación |
| **Series de tiempo** | `SARIMAX`, `adfuller` (statsmodels) | Pronóstico de precios a 12 meses con captura de estacionalidad |
| **Visualización** | `matplotlib`, `seaborn` | Gráficas exploratorias y de resultados |
| **Costos logísticos** | `COSTO_BASE_TRANSPORTE = 0.90` | Costo proxy (MXN/unidad) dentro del mismo estado |
| | `RECARGO_INTERESTATAL = 0.45` | Recargo adicional por cruzar estado de origen |
| | `COSTO_OPERATIVO = 0.25` | Manejo y mano de obra por unidad |
| | `MARGEN_MINIMO = 0.20` | Umbral mínimo de ganancia para considerar una ruta viable |
| **Pronóstico** | `HORIZONTE_MESES = 12` | Ventana de predicción hacia adelante en meses |

## Inicialización de SparkSession

La `SparkSession` es el punto de entrada unificado a todas las funcionalidades de Spark.
Al correr en modo local dentro de Docker, el driver y los executors conviven en el mismo proceso
JVM, por lo que la configuración de memoria es especialmente relevante para el rendimiento.

| Parámetro | Valor | Descripción |
|---|---|---|
| `.appName()` | `"MercadoJusto"` | Nombre que identifica la aplicación en la Spark UI (`localhost:4040`) |
| `spark.sql.shuffle.partitions` | `8` | Particiones al reorganizar datos. El default es 200; reducirlo es adecuado para datasets medianos en local, evitando overhead innecesario |
| `spark.driver.memory` | `2g` | RAM asignada al proceso driver, que coordina la ejecución y recolecta resultados con `.collect()` |
| `.getOrCreate()` | — | Reutiliza una sesión existente si ya hay una activa; de lo contrario crea una nueva |
| `.setLogLevel("ERROR")` | `"ERROR"` | Suprime mensajes INFO y WARN, mostrando solo errores críticos para mantener la salida limpia |

In [1]:
import warnings, re, unicodedata
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# PySpark ─────────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType, IntegerType
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline

# Series de tiempo (statsmodels no tiene versión nativa Spark; se usa con collect)
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

# ── Parámetros ajustables ────────────────────────────────────────────────────
CSV_PATH = Path("sniim_resultados.csv")
OVERRIDE_PRESENTACION: dict = {}

# Costos logísticos proxy (MXN / unidad de presentación)
COSTO_BASE_TRANSPORTE = 0.90   # mismo estado
RECARGO_INTERESTATAL  = 0.45   # estado diferente
COSTO_OPERATIVO       = 0.25   # manejo y mano de obra
MARGEN_MINIMO         = 0.20   # umbral mínimo de ganancia

HORIZONTE_MESES = 12  # meses a pronosticar hacia adelante

# ── Inicializar SparkSession ─────────────────────────────────────────────────
spark = (SparkSession.builder
         .appName("MercadoJusto")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.driver.memory", "2g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} iniciado ✓")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/01 19:57:32 WARN Utils: Your hostname, nictez-Nitro-AN515-55, resolves to a loopback address: 127.0.1.1; using 192.168.0.13 instead (on interface wlp0s20f3)
26/06/01 19:57:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/01 19:57:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/01 19:57:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.1.1 iniciado ✓


---

# Carga y limpieza de datos {#sec-datos}

In [2]:
def norm(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    t = unicodedata.normalize("NFKD", str(v).strip().lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", t)

def parse_kg(p):
    t = norm(p)
    if t is None: return None
    if t in {"kilogramo", "kilo", "kg"}: return 1.0
    if "kg" in t or "kilogramo" in t:
        m = re.search(r"(\d+(?:[.,]\d+)?)", t)
        return float(m.group(1).replace(",", ".")) if m else 1.0
    m = re.search(r"(\d+(?:[.,]\d+)?)", t)
    if m and any(w in t for w in ["caja","arpilla","bolsa","costal","malla","saco"]):
        return float(m.group(1).replace(",", "."))
    return None

def estado_destino(d):
    if d is None: return None
    return str(d).split(":")[0].strip() if ":" in str(d) else str(d).strip()

def costo_trans(o, d):
    return COSTO_BASE_TRANSPORTE if norm(o) == norm(d) \
           else COSTO_BASE_TRANSPORTE + RECARGO_INTERESTATAL

In [3]:
# ── Registrar UDFs ───────────────────────────────────────────────────────────
norm_udf          = F.udf(norm,           StringType())
parse_kg_udf      = F.udf(parse_kg,       DoubleType())
estado_dest_udf   = F.udf(estado_destino, StringType())
costo_trans_udf   = F.udf(lambda o, d: float(costo_trans(o, d)), DoubleType())

# ── Leer CSV con Spark ───────────────────────────────────────────────────────
# Se lee con quote='"' y escape='"' para tolerar comillas mal cerradas en el CSV.
# Después se eliminan comillas residuales en las columnas de texto clave.
raw_spark = (spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .option("encoding", "UTF-8")
             .option("multiLine", "false")
             .option("quote", '"')
             .option("escape", '"')
             .option("mode", "PERMISSIVE")
             .csv(str(CSV_PATH)))

# Normalizar nombres de columnas
def norm_col(c):
    t = unicodedata.normalize("NFKD", c.strip().lower())
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    return re.sub(r"[^a-z0-9]+", "_", t).strip("_")

rename_map = {c: norm_col(c) for c in raw_spark.columns}
df_spark = raw_spark
for old, new in rename_map.items():
    if old != new:
        df_spark = df_spark.withColumnRenamed(old, new)

# ── Limpiar comillas residuales en columnas de texto ─────────────────────────
# Algunos valores como '"nuevo leon' o '"sonora' tienen comillas iniciales
# porque el CSV contiene comillas sin cerrar. Se eliminan aquí de forma global.
for _tc in ["origen", "destino", "producto", "presentacion"]:
    if _tc in df_spark.columns:
        df_spark = df_spark.withColumn(
            _tc,
            F.regexp_replace(F.col(_tc).cast(StringType()), "[\x22\x27]", "")
        )

# ── Transformaciones ─────────────────────────────────────────────────────────
# Parsear fecha (intenta formato dd/MM/yyyy y yyyy-MM-dd)
# Los precios pueden venir con coma de miles: '1,000.00' → quitar coma antes del cast
def clean_num_col(df, col_name):
    """Elimina comas de miles y hace cast a Double."""
    return df.withColumn(
        col_name,
        F.regexp_replace(F.col(col_name).cast(StringType()), ",", "").cast(DoubleType())
    )

df_spark = (df_spark
    .withColumn("fecha",
        F.coalesce(
            F.to_date(F.col("fecha"), "dd/MM/yyyy"),
            F.to_date(F.col("fecha"), "yyyy-MM-dd"),
            F.to_date(F.col("fecha"), "MM/dd/yyyy"),
        ))
)
for _c in ["precio_min", "precio_max", "precio_frec"]:
    df_spark = clean_num_col(df_spark, _c)

# precio_central = precio_frec ó promedio(min,max)
df_spark = df_spark.withColumn(
    "precio_central",
    F.coalesce(
        F.col("precio_frec"),
        (F.col("precio_min") + F.col("precio_max")) / 2.0
    )
)

df_spark = (df_spark
    .withColumn("presentacion_kg", parse_kg_udf(F.col("presentacion")))
    .withColumn("destino_estado",  estado_dest_udf(F.col("destino")))
    .withColumn("origen_estado",   F.col("origen").cast(StringType()))
    .withColumn("mes",             F.month(F.col("fecha")))
    .withColumn("anio_mes",        F.date_format(F.col("fecha"), "yyyy-MM"))
)

# Filtrar nulos y precios inválidos
df_spark = (df_spark
    .dropDuplicates()
    .dropna(subset=["fecha", "producto", "origen", "destino"])
    .filter(F.col("precio_central").isNotNull() & (F.col("precio_central") > 0))
)

# ── Presentación dominante por producto ─────────────────────────────────────
from pyspark.sql.window import Window

win_prod = Window.partitionBy("producto")
pres_counts = (df_spark
    .groupBy("producto", "presentacion")
    .agg(F.count("*").alias("cnt"))
)
win_pres = Window.partitionBy("producto").orderBy(F.col("cnt").desc())
dominante_spark = (pres_counts
    .withColumn("rn", F.row_number().over(win_pres))
    .filter(F.col("rn") == 1)
    .select("producto", F.col("presentacion").alias("pres_dominante"))
)

df_spark = df_spark.join(dominante_spark, on="producto", how="left")

# Aplicar overrides
if OVERRIDE_PRESENTACION:
    from pyspark.sql.functions import create_map, lit
    override_map = create_map([lit(x) for pair in OVERRIDE_PRESENTACION.items() for x in pair])
    df_spark = df_spark.withColumn(
        "pres_obj",
        F.coalesce(override_map[F.col("producto")], F.col("pres_dominante"))
    )
else:
    df_spark = df_spark.withColumn("pres_obj", F.col("pres_dominante"))

# Filtrar a presentación objetivo
df_a = (df_spark
    .filter(F.col("presentacion") == F.col("pres_obj"))
    .withColumn("precio_modelo", F.col("precio_central"))
)
df_a.cache()

productos = [r["producto"] for r in df_a.select("producto").distinct().collect()]

# ── Resumen ──────────────────────────────────────────────────────────────────
total_raw   = df_spark.count()
total_anal  = df_a.count()
n_productos = df_a.select("producto").distinct().count()
fecha_min   = df_a.agg(F.min("fecha")).collect()[0][0]
fecha_max   = df_a.agg(F.max("fecha")).collect()[0][0]

spark.createDataFrame([
    ("Registros totales",       f"{total_raw:,}"),
    ("Registros en análisis",   f"{total_anal:,}"),
    ("Productos",               str(n_productos)),
    ("Fecha inicio",            str(fecha_min)),
    ("Fecha fin",               str(fecha_max)),
], ["Métrica", "Valor"]).toPandas().set_index("Métrica")

UnsupportedOperationException: getSubject is not supported

In [ ]:
print("Presentación utilizada por producto\n")
dom_rows = dominante_spark.collect()
filas_pres = []
for row in dom_rows:
    prod = row["producto"]
    filas_pres.append({
    "Producto": prod,
    "Presentación utilizada": OVERRIDE_PRESENTACION.get(
        prod,
        row["pres_dominante"]
    )
})
spark.createDataFrame(filas_pres).toPandas()

---

# Herramientas avanzadas – Análisis exploratorio {#sec-eda}

## Estadísticos descriptivos

In [ ]:
stats_cols = ["precio_min", "precio_max", "precio_frec", "precio_central"]
desc = (df_a
    .groupBy("producto")
    .agg(*[
        F.round(F.mean(c),    2).alias(f"{c}_mean")   for c in stats_cols
    ] + [
        F.round(F.stddev(c),  2).alias(f"{c}_std")    for c in stats_cols
    ] + [
        F.round(F.min(c),     2).alias(f"{c}_min")    for c in stats_cols
    ] + [
        F.round(F.max(c),     2).alias(f"{c}_max")    for c in stats_cols
    ])
)
desc.toPandas()

## Distribución de precios por producto

In [ ]:
colores = ["#264653", "#2a9d8f"]
fig, axes = plt.subplots(len(productos), 1, figsize=(10, 8))
for ax, prod, color in zip(axes, productos, colores):
    vals = (df_a.filter(F.col("producto") == prod)
               .select("precio_modelo")
               .dropna()
               .toPandas()["precio_modelo"])
    ax.hist(vals, bins=40, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(vals.mean(), color="#e76f51", linestyle="--", linewidth=1.5,
               label=f"Media: ${vals.mean():.2f}")
    ax.axvline(vals.median(), color="#f4a261", linestyle=":", linewidth=1.5,
               label=f"Mediana: ${vals.median():.2f}")
    ax.set_title(prod, fontsize=10)
    ax.set_xlabel("Precio (MXN)")
    ax.set_ylabel("Frecuencia")
    ax.legend(fontsize=8)
plt.suptitle("Distribución de precios por producto (presentación dominante)", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

## Serie histórica de precios por producto

> Precio promedio mensual. Cada punto es un mes; la línea conecta la evolución histórica de 2000 a 2026.

In [ ]:
serie_m = (df_a
    .groupBy("anio_mes", "producto")
    .agg(F.mean("precio_modelo").alias("precio_promedio"))
    .withColumn("fecha_mes", F.to_date(F.concat_ws("-", F.col("anio_mes"), F.lit("01"))))
    .orderBy("fecha_mes")
    .toPandas()
)

colores_prod = {productos[0]: "#264653", productos[1]: "#2a9d8f"}

fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=False)
for ax, prod in zip(axes, productos):
    grp   = serie_m[serie_m["producto"] == prod].sort_values("fecha_mes")
    color = colores_prod.get(prod, "#e76f51")
    ax.plot(grp["fecha_mes"], grp["precio_promedio"],
            marker="o", markersize=3, linewidth=1.5, color=color)
    ax.fill_between(grp["fecha_mes"], grp["precio_promedio"],
                    grp["precio_promedio"].min(), alpha=0.08, color=color)
    pres_obj_val = OVERRIDE_PRESENTACION.get(prod,
        [r["pres_dominante"] for r in dom_rows if r["producto"] == prod][0])
    ax.set_title(f"{prod}  –  Presentación: {pres_obj_val}", fontsize=11)
    ax.set_ylabel("Precio promedio (MXN)")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0f"))
    ax.set_xlabel("Mes")
plt.suptitle("Evolución histórica mensual de precios (2000–2026)", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## Top orígenes y destinos

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 10))

for ax, col, color, title in [
    (axes[0], "origen_estado", "#457b9d", "Top 10 orígenes"),
    (axes[1], "destino_estado", "#e9c46a", "Top 10 destinos"),
]:

    total = df_a.count()

    tmp = (
        df_a.groupBy(col)
            .count()
            .orderBy(F.col("count").desc())
            .limit(10)
            .toPandas()
    )

    tmp["porcentaje"] = tmp["count"] / total * 100
    tmp = tmp.sort_values("count")

    bars = ax.barh(tmp[col], tmp["count"], color=color)

    ax.bar_label(
        bars,
        labels=[
            f"{c:,.0f} ({p:.1f}%)"
            for c, p in zip(tmp["count"], tmp["porcentaje"])
        ],
        padding=3,
        fontsize=8
    )

    ax.set_title(title)
    ax.set_xlabel("Registros")

plt.tight_layout()
plt.show()

---

# Modelos estocásticos – Pronóstico SARIMA {#sec-arima}

> Se usa una **serie mensual** (precio promedio por mes) para capturar la estacionalidad anual. El modelo **SARIMA(1,1,1)(0,1,1,12)** aplica diferenciación estacional de 12 meses, produciendo pronósticos con variación realista — no lineal. Este pronóstico alimenta directamente el modelo de ML.

In [ ]:
import pandas as pd   # solo para statsmodels (series de tiempo locales)

ORDEN_ARIMA    = (1, 1, 1)
ORDEN_SEASONAL = (0, 1, 1, 12)

resultados_arima = {}

for prod in productos:
    # Colectar serie mensual (statsmodels opera localmente)
    serie_pd = (df_a
        .filter(F.col("producto") == prod)
        .groupBy("anio_mes")
        .agg(F.mean("precio_modelo").alias("precio_promedio"))
        .orderBy("anio_mes")
        .toPandas()
    )
    serie_pd["fecha"] = pd.to_datetime(serie_pd["anio_mes"] + "-01")
    serie_pd = serie_pd.set_index("fecha")["precio_promedio"]
    serie    = serie_pd.resample("MS").mean().interpolate(method="time").ffill().bfill()

    n_test = 12
    train, test = serie.iloc[:-n_test], serie.iloc[-n_test:]

    mod_eval = SARIMAX(train, order=ORDEN_ARIMA, seasonal_order=ORDEN_SEASONAL,
                       enforce_stationarity=False,
                       enforce_invertibility=False).fit(disp=False)
    pred_test = mod_eval.get_forecast(steps=n_test)
    pred_mean = pred_test.predicted_mean
    pred_ci   = pred_test.conf_int()

    mae  = float(np.mean(np.abs(test.values - pred_mean.values)))
    rmse = float(np.sqrt(np.mean((test.values - pred_mean.values)**2)))
    mape = float(np.mean(np.abs((test.values - pred_mean.values) / test.values)) * 100)

    mod_full = SARIMAX(serie, order=ORDEN_ARIMA, seasonal_order=ORDEN_SEASONAL,
                       enforce_stationarity=False,
                       enforce_invertibility=False).fit(disp=False)
    fc_fut      = mod_full.get_forecast(steps=HORIZONTE_MESES)
    fc_fut_mean = fc_fut.predicted_mean
    fc_fut_ci   = fc_fut.conf_int()
    fut_idx     = pd.date_range(serie.index[-1] + pd.DateOffset(months=1),
                                periods=HORIZONTE_MESES, freq="MS")
    fc_fut_mean.index = fut_idx
    fc_fut_ci.index   = fut_idx

    resultados_arima[prod] = {
        "serie": serie, "train": train, "test": test,
        "pred_mean": pred_mean, "pred_ci": pred_ci,
        "fc_fut_mean": fc_fut_mean, "fc_fut_ci": fc_fut_ci,
        "precio_futuro": float(fc_fut_mean.mean()),
        "mae": mae, "rmse": rmse, "mape": mape,
    }
    print(f"{'='*55}")
    print(f"Producto : {prod}")
    print(f"Modelo   : SARIMA{ORDEN_ARIMA}{ORDEN_SEASONAL}")
    print(f"Obs. mensuales: {len(serie)}  |  Prueba: {n_test} meses")
    print(f"MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.1f}%")
    print(f"Precio prom. pronosticado ({HORIZONTE_MESES}m): ${float(fc_fut_mean.mean()):.2f}")

## Métricas de ajuste

In [ ]:
metricas_rows = [
    {"Producto": p,
     "MAE": round(float(v["mae"]),2), "RMSE": round(float(v["rmse"]),2),
     "MAPE (%)": round(float(v["mape"]),1),
     "Precio futuro prom.": f'${v["precio_futuro"]:.2f}'}
    for p, v in resultados_arima.items()
]
pd.DataFrame(metricas_rows)

## Pronóstico vs. real – evaluación en prueba

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 10))
for ax, prod, color in zip(axes, productos, ["#264653","#2a9d8f"]):
    r = resultados_arima[prod]
    ax.plot(r["train"].index, r["train"].values,
            color=color, linewidth=1.2, label="Entrenamiento")
    ax.plot(r["test"].index, r["test"].values,
            color="#1d3557", linewidth=1.5, label="Real (prueba)")
    ax.plot(r["pred_mean"].index, r["pred_mean"].values,
            color="#e76f51", linewidth=1.8, linestyle="--", label="SARIMA (prueba)")
    ax.fill_between(r["pred_ci"].index,
                    r["pred_ci"].iloc[:,0], r["pred_ci"].iloc[:,1],
                    color="#e76f51", alpha=0.15, label="IC 95%")
    ax.set_title(f"{prod}  |  MAE={r['mae']:.2f}  MAPE={r['mape']:.1f}%", fontsize=11)
    ax.set_ylabel("Precio (MXN)"); ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0f"))
plt.suptitle("SARIMA – Evaluación en prueba (últimos 12 meses)", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## Pronóstico operativo – próximos {python} `HORIZONTE_MESES` meses

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 10))
for ax, prod, color in zip(axes, productos, ["#264653","#2a9d8f"]):
    r       = resultados_arima[prod]
    ultimos = r["serie"].iloc[-24:]
    fc_m    = r["fc_fut_mean"]
    fc_ci   = r["fc_fut_ci"]

    ax.plot(ultimos.index, ultimos.values,
            color=color, linewidth=1.5, label="Histórico (24m)")
    ax.plot(fc_m.index, fc_m.values,
            color="#e76f51", linewidth=2, marker="o", markersize=4,
            label=f"Pronóstico ({HORIZONTE_MESES}m)")
    ax.fill_between(fc_ci.index, fc_ci.iloc[:,0], fc_ci.iloc[:,1],
                    color="#e76f51", alpha=0.15, label="IC 95%")
    ax.axvline(r["serie"].index[-1], color="gray", linestyle=":", linewidth=1)
    ax.set_title(
        f"{prod}  |  Precio prom. pronosticado: ${r['precio_futuro']:.2f}", fontsize=11)
    ax.set_ylabel("Precio (MXN)"); ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0f"))
plt.suptitle(f"Pronóstico operativo SARIMA – próximos {HORIZONTE_MESES} meses",
             fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---

# Machine Learning – ¿Conviene vender? {#sec-ml}

> El **Random Forest** aprende a identificar momentos de precio favorable basándose en criterios históricos por ruta — no en la misma fórmula que usaría para predecir — evitando así el *label leakage*.

## Etiqueta histórica por ruta

La etiqueta `vende = 1` se define con **dos criterios alternativos** calculados en Spark con ventanas y agregaciones, sin tocar el precio pronosticado:

| Criterio | Definición |
|---|---|
| **Cuantil de precio** | `y = 1` si el precio actual supera el percentil 70 histórico de esa ruta (origen–destino–producto). El precio está en el **top 30%** de su historia → momento favorable. |
| **Margen histórico** | `y = 1` si el margen actual (precio actual − costos reales de transporte − costo operativo) supera el **margen promedio histórico** de esa misma ruta. |

La etiqueta final combina ambos: `vende = 1` si **al menos uno** de los dos criterios se cumple. Por construcción, el criterio de margen activa en aproximadamente el 50% de los registros (mitad por encima del promedio), lo que garantiza un balance razonable de clases. El umbral de decisión del modelo se ajusta a **0.35** (en lugar del 0.5 por defecto) para compensar la variabilidad natural entre rutas con pocas observaciones.

Las features del modelo son características **observables en el momento de la decisión**: precio actual, costo de transporte, mes, si es ruta intraestatal, y el precio pronosticado por SARIMA como señal de contexto de mercado (no como parte del cálculo de la etiqueta).

## Construcción del panel en Spark

In [ ]:
from pyspark.sql import Row
from functools import reduce
from pyspark.sql import DataFrame as SparkDF
from pyspark.sql.window import Window

# Parámetro: percentil de precio por encima del cual se considera "buen momento"
CUANTIL_PRECIO = 0.70   # top 30% de la historia de esa ruta

panel_parts = []
for prod in productos:
    pf = resultados_arima[prod]["precio_futuro"]   # señal SARIMA (solo como feature)

    sub = (df_a
        .filter(F.col("producto") == prod)
        .withColumn("precio_actual",       F.col("precio_modelo"))
        .withColumn("precio_pronosticado", F.lit(float(pf)))   # feature, NO entra en label
        .withColumn("mismo_estado",
            (norm_udf(F.col("origen_estado")) == norm_udf(F.col("destino_estado")))
            .cast(IntegerType()))
        .withColumn("costo_transporte",
            costo_trans_udf(F.col("origen_estado"), F.col("destino_estado")))
        # Margen con precio ACTUAL (no pronosticado) — para el criterio histórico
        .withColumn("margen_actual",
            F.col("precio_actual") - F.col("costo_transporte") - F.lit(COSTO_OPERATIVO))
    )

    # ── Ventana por ruta (producto + origen–destino) ────────────────────────
    # Incluir producto evita que rutas con el mismo O-D pero distinto producto
    # compartan percentiles diferentes
    win_ruta = Window.partitionBy("producto", "origen_estado", "destino_estado")

    sub = (sub
        # Criterio 1: precio actual > percentil 70 histórico de la ruta
        .withColumn("umbral_precio",
            F.percentile_approx("precio_actual", CUANTIL_PRECIO).over(win_ruta))
        .withColumn("criterio_cuantil",
            (F.col("precio_actual") >= F.col("umbral_precio")).cast(IntegerType()))

        # Criterio 2: margen actual > margen promedio histórico de la ruta
        .withColumn("margen_prom_ruta",
            F.avg("margen_actual").over(win_ruta))
        .withColumn("criterio_margen",
            (F.col("margen_actual") >= F.col("margen_prom_ruta")).cast(IntegerType()))

        # Etiqueta final: al menos uno de los dos criterios se cumple
        .withColumn("vende",
            F.greatest(F.col("criterio_cuantil"), F.col("criterio_margen")))
    )
    panel_parts.append(sub)

panel = reduce(SparkDF.union, panel_parts)
panel.cache()

print(f"Registros en panel: {panel.count():,}")
print(f"Percentil de precio usado como umbral: {CUANTIL_PRECIO:.0%}")
print("\nDistribución de la etiqueta 'vende' por producto:")
panel.groupBy("producto", "vende").count().orderBy("producto","vende").show()

print("\nBalance de criterios (% de registros donde aplica cada uno):")
panel.agg(
    F.round(F.mean("criterio_cuantil")*100, 1).alias("% criterio_cuantil"),
    F.round(F.mean("criterio_margen")*100,  1).alias("% criterio_margen"),
    F.round(F.mean("vende")*100,            1).alias("% vende (union)"),
).show()

## Entrenamiento del Random Forest

In [ ]:
# Features observables en el momento de la decisión:
#   precio_actual       → precio de mercado en ese momento
#   precio_pronosticado → señal SARIMA como contexto (no parte de la etiqueta)
#   margen_actual       → margen con precio real (precio_actual − costos)
#   costo_transporte    → costo logístico de la ruta
#   mismo_estado        → flag ruta intraestatal
#   mes                 → estacionalidad
#   umbral_precio       → contexto histórico de la ruta (percentil calculado en Spark)
#   margen_prom_ruta    → contexto histórico de margen de la ruta
FEATS = ["precio_actual", "precio_pronosticado", "margen_actual",
         "costo_transporte", "mismo_estado", "mes",
         "umbral_precio", "margen_prom_ruta"]

# Split temporal 80/20 (respeta orden cronológico para evitar data leakage temporal)
fechas_sorted = [r[0] for r in panel.select("fecha").distinct().orderBy("fecha").collect()]
corte_idx = int(len(fechas_sorted) * 0.8)
corte_fecha = fechas_sorted[corte_idx]

train_sp = panel.filter(F.col("fecha") <= corte_fecha)
test_sp  = panel.filter(F.col("fecha") >  corte_fecha)

# Rellenar nulos con 0 en features
for f in FEATS:
    train_sp = train_sp.fillna({f: 0})
    test_sp  = test_sp.fillna({f: 0})

# Assembler + RandomForest Pipeline
assembler = VectorAssembler(inputCols=FEATS, outputCol="features", handleInvalid="skip")

# Calcular pesos para balancear clases (evita que el RF prediga todo como 0
# cuando la clase mayoritaria domina el entrenamiento)
n_total = float(train_sp.count())
n_pos   = float(train_sp.filter(F.col("vende") == 1).count())
n_neg   = n_total - n_pos
w_pos   = n_total / (2.0 * n_pos)  if n_pos > 0 else 1.0
w_neg   = n_total / (2.0 * n_neg)  if n_neg > 0 else 1.0
print(f"Balance de clases en train — vende=1: {n_pos:.0f} ({100*n_pos/n_total:.1f}%)  vende=0: {n_neg:.0f} ({100*n_neg/n_total:.1f}%)")
print(f"Pesos aplicados — w(vende=1): {w_pos:.3f}  w(vende=0): {w_neg:.3f}")

train_sp = (train_sp
    .withColumn("peso", F.when(F.col("vende") == 1, F.lit(w_pos)).otherwise(F.lit(w_neg))))
test_sp  = test_sp.withColumn("peso", F.lit(1.0))   # peso neutro en test

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="vende",
    weightCol="peso",          # balanceo de clases por peso
    numTrees=150,
    maxDepth=8,
    minInstancesPerNode=5,     # evita hojas con muy pocas observaciones
    seed=42
)

pipeline = Pipeline(stages=[assembler, rf])
modelo_rf = pipeline.fit(train_sp)

# Predicciones
pred_test_sp = modelo_rf.transform(test_sp)

# Métricas
eval_acc = MulticlassClassificationEvaluator(
    labelCol="vende", predictionCol="prediction", metricName="accuracy")
eval_f1  = MulticlassClassificationEvaluator(
    labelCol="vende", predictionCol="prediction", metricName="f1")
eval_prec = MulticlassClassificationEvaluator(
    labelCol="vende", predictionCol="prediction", metricName="weightedPrecision")
eval_rec  = MulticlassClassificationEvaluator(
    labelCol="vende", predictionCol="prediction", metricName="weightedRecall")
eval_auc  = BinaryClassificationEvaluator(
    labelCol="vende", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

acc   = eval_acc.evaluate(pred_test_sp)
f1    = eval_f1.evaluate(pred_test_sp)
prec  = eval_prec.evaluate(pred_test_sp)
rec   = eval_rec.evaluate(pred_test_sp)
auc   = eval_auc.evaluate(pred_test_sp)

pd.DataFrame([
    {"Métrica": "Accuracy",          "Valor": round(float(acc),  4)},
    {"Métrica": "Precisión (pond.)", "Valor": round(float(prec), 4)},
    {"Métrica": "Recall (pond.)",    "Valor": round(float(rec),  4)},
    {"Métrica": "F1 (pond.)",        "Valor": round(float(f1),   4)},
    {"Métrica": "AUC-ROC",           "Valor": round(float(auc),  4)},
])

In [ ]:
from sklearn.metrics import confusion_matrix

# Recolectar para graficar
y_true = [r["vende"]      for r in pred_test_sp.select("vende","prediction").collect()]
y_pred = [int(r["prediction"]) for r in pred_test_sp.select("vende","prediction").collect()]

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_xticklabels(["No vender","Vender"])
ax.set_yticklabels(["No vender","Vender"], rotation=0)
ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
ax.set_title("Matriz de confusión – Random Forest")
plt.tight_layout(); plt.show()

## Importancia de variables

In [ ]:
rf_model      = modelo_rf.stages[-1]
importancias  = rf_model.featureImportances.toArray()

fig, ax = plt.subplots(figsize=(9, 4))
sorted_idx = np.argsort(importancias)
bars = ax.barh([FEATS[i] for i in sorted_idx], importancias[sorted_idx], color="#2a9d8f")
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=9)
ax.set_title("Importancia de variables – Random Forest", fontsize=12)
ax.set_xlabel("Importancia (Gini)")
plt.tight_layout(); plt.show()

## Tabla de recomendación: Origen × Destino

> **Lectura:** cada fila es un estado de **origen**, cada columna un estado de **destino**. La celda indica si el Random Forest recomienda vender — aprendido de criterios históricos por ruta (precio en top 30% y/o margen sobre el promedio histórico), no de una fórmula fija.

In [ ]:
# Aplicar predicción al panel completo
panel_pred = modelo_rf.transform(panel.fillna({f: 0 for f in FEATS}))
panel_pred = panel_pred.withColumn("pred_vende", F.col("prediction").cast(IntegerType()))
panel_pred.cache()

for prod in productos:
    sub_p = panel_pred.filter(F.col("producto") == prod)
    pivot_pd = (sub_p
        .groupBy("origen_estado", "destino_estado")
        .agg(F.mean("pred_vende").alias("prob_vende"))
        .withColumn("recomendacion",
            F.when(F.col("prob_vende") >= 0.35, "Vender").otherwise("No vender"))
        .toPandas()
        .pivot(index="origen_estado", columns="destino_estado", values="recomendacion")
        .fillna("—")
    )
    print(f"\n{'='*60}")
    print(f"PRODUCTO: {prod}")
    print(f"Precio pronosticado SARIMA (prom. {HORIZONTE_MESES}m): "
          f"${resultados_arima[prod]['precio_futuro']:.2f}")
    print(f"{'='*60}\n")
    
    # Renderizar tabla con scroll horizontal interno para evitar romper el layout de la página
    from IPython.display import display, HTML
    style = f"""
    <div style="overflow-x: auto; max-width: 100%; border: 1px solid #e2e8f0; border-radius: 8px; margin-bottom: 2em;">
        <style>
            .df-{prod.replace(' ', '-')} {{ width: 100% !important; border-collapse: collapse !important; font-size: 11px !important; }}
            .df-{prod.replace(' ', '-')} th, .df-{prod.replace(' ', '-')} td {{ padding: 8px !important; text-align: center !important; border: 1px solid #f1f5f9 !important; }}
            .df-{prod.replace(' ', '-')} th {{ background-color: #f8fafc !important; color: #475569 !important; }}
        </style>
    """
    display(HTML(style + pivot_pd.to_html(classes=f"df-{prod.replace(' ', '-')}") + "</div>"))

---

# Investigación de Operaciones – Optimización Logística {#sec-dijkstra}

Esta sección detalla la lógica algorítmica que fundamenta nuestra solución de transporte. A diferencia de una ruta estática, el sistema permite resolver automáticamente la secuencia óptima de entrega y la estimación de costos generada mediante algoritmos de optimización logística basados en **Dijkstra** y análisis de rutas.

## Dashboard Interactivo de Planificación

Para facilitar el uso de estas herramientas avanzadas de optimización, hemos desarrollado un **Dashboard Interactivo** independiente. En este sitio web, el productor podrá:

1. Seleccionar sus productos y puntos de origen.
2. Añadir múltiples paradas de destino para simular sus rutas de entrega.
3. Obtener en tiempo real la comparativa de costos y la secuencia óptima de viaje generada por el algoritmo de Dijkstra.

Haz clic en el siguiente enlace para acceder a la herramienta de planificación:

<a href="https://mercado-justo-beta.vercel.app/logistica"
   target="_blank"
   class="btn btn-primary">
   Abrir Dashboard Interactivo
</a>

*(Nota: Este link redirige a la aplicación web externa donde se encuentra desplegada la interfaz interactiva).*

---

## Conexión con el Modelo Predictivo

A diferencia de un cálculo de distancia pura (como Google Maps), este motor integra las señales del **Random Forest** entrenado anteriormente. Cuando una ruta tiene una alta probabilidad de éxito comercial, el algoritmo "recompensa" ese tramo reduciendo artificialmente su peso en el grafo. Esto asegura que la recomendación no sea solo la más corta en kilómetros, sino la más eficiente financieramente para el productor.

---

# Plan de Marketing Digital {#sec-marketing}

> Las acciones de marketing se activan directamente con los resultados del pipeline: pronóstico favorable → mensaje personalizado → ruta óptima comunicada.

In [ ]:
filas_mkt = []
for prod in productos:
    r     = resultados_arima[prod]
    sub_p = panel_pred.filter(F.col("producto") == prod)

    p_act  = sub_p.agg(F.percentile_approx("precio_actual", 0.5)).collect()[0][0]
    p_pron = r["precio_futuro"]

    n_rutas = int(sub_p
        .groupBy("origen_estado","destino_estado")
        .agg(F.mean("pred_vende").alias("prob"))
        .filter(F.col("prob") >= 0.35)
        .count())

    mejor_row = (sub_p
        .filter(F.col("pred_vende") == 1)
        .groupBy("origen_estado","destino_estado")
        .agg(F.mean("margen_actual").alias("margen_prom"))
        .orderBy(F.col("margen_prom").desc())
        .limit(1)
        .collect())

    mejor_str = (f"{mejor_row[0]['origen_estado']} → {mejor_row[0]['destino_estado']}"
                 if mejor_row else "—")

    filas_mkt.append({
        "Producto": prod,
        "Precio actual (med.)": f"${p_act:.2f}" if p_act else "N/D",
        "Precio pron. SARIMA": f"${p_pron:.2f}",
        "Mejor ruta": mejor_str,
        "Rutas recomendadas": n_rutas,
    })
spark.createDataFrame(filas_mkt).toPandas()

## Tabla de acciones

In [ ]:
acciones = spark.createDataFrame([
    {"Acción": "Alerta de precio favorable",   "Objetivo": "Captar atención del productor",
     "Canal": "WhatsApp Business / Facebook",   "Cuándo activar": "Pronóstico > precio actual + costos",
     "KPI": "Tasa de apertura"},
    {"Acción": "Mensaje segmentado por ruta",   "Objetivo": "Personalizar la comunicación",
     "Canal": "Email / WhatsApp",               "Cuándo activar": "Probabilidad de venta > 50%",
     "KPI": "Respuestas / clics"},
    {"Acción": "Mapa de rutas recomendadas",    "Objetivo": "Convertir la decisión de venta",
     "Canal": "Dashboard HTML (este notebook)", "Cuándo activar": f"Margen estimado > ${MARGEN_MINIMO}",
     "KPI": "Ventas concretadas"},
    {"Acción": "Pronóstico mensual",            "Objetivo": "Reducir incertidumbre",
     "Canal": "Correo electrónico",             "Cuándo activar": "Cada inicio de mes",
     "KPI": "Retención mensual"},
    {"Acción": "Testimonio de productor",       "Objetivo": "Generar confianza en el modelo",
     "Canal": "Redes sociales",                 "Cuándo activar": "Al cerrar venta exitosa",
     "KPI": "Engagement"},
    {"Acción": "Tablero Quarto actualizado",    "Objetivo": "Sostener seguimiento continuo",
     "Canal": "Quarto HTML publicado",          "Cuándo activar": "Con cada actualización del CSV",
     "KPI": "Visitas al tablero"},
])
acciones.toPandas()

## Mensaje ejemplo – WhatsApp Business

In [ ]:
for prod in productos:
    r     = resultados_arima[prod]
    sub_p = panel_pred.filter(F.col("producto") == prod)

    n_v = sub_p.groupBy("origen_estado","destino_estado").agg(
        F.mean("pred_vende").alias("prob")).filter(F.col("prob")>=0.5).count()

    p_act = sub_p.agg(F.percentile_approx("precio_actual", 0.5)).collect()[0][0] or 0

    mejor_row = (sub_p.filter(F.col("pred_vende")==1)
        .groupBy("origen_estado","destino_estado")
        .agg(F.mean("margen_actual").alias("m"))
        .orderBy(F.col("m").desc()).limit(1).collect())
    mejor = (mejor_row[0]["origen_estado"], mejor_row[0]["destino_estado"]) \
            if mejor_row else ("—", "—")

    print(f"""
{'─'*55}
ALERTA MERCADO JUSTO – {prod}
{'─'*55}
   Pronóstico próximos {HORIZONTE_MESES} meses (SARIMA):
   Precio actual referencia : ${p_act:.2f}
   Precio pronosticado      : ${r['precio_futuro']:.2f}

 Rutas convenientes detectadas : {n_v}
 Mejor ruta : {mejor[0]} → {mejor[1]}

Consulta la tabla completa de rutas en el tablero.
{'─'*55}
""")

---

# Conclusiones

El pipeline conecta seis etapas donde la salida de cada una alimenta a la siguiente:

| Etapa | Herramienta | Salida para la siguiente etapa |
|---|---|---|
| **Scraping** | Web Scraping interactivo (BeautifulSoup) | Archivo CSV actualizado (SNIIM) |
| **EDA** | Estadísticos + gráficas históricas (Spark) | Serie mensual limpia por producto |
| **SARIMA** | Modelo estocástico estacional | Precio pronosticado por producto |
| **Random Forest** | ML supervisado (PySpark MLlib) | Tabla Origen × Destino (vender / no vender) |
| **Dijkstra** | Teoría de grafos (Ruta más corta) | Ruta logística óptima de menor costo |
| **Marketing** | Plan accionable | Mensajes y alertas basadas en datos |

**Decisiones técnicas clave:**

- **PySpark:** toda la carga, limpieza, agregaciones y el modelo de ML corren en el motor distribuido; solo statsmodels opera localmente sobre series mensuales colectadas (< 200 puntos por producto).
- **Serie mensual:** elimina ruido diario y permite capturar estacionalidad anual.
- **SARIMA(1,1,1)(0,1,1,12):** la diferenciación estacional produce pronósticos con variación realista, no lineal.
- **Random Forest (100 árboles, profundidad 8):** mayor robustez frente a overfitting que un árbol individual; la importancia de variables cuantifica qué factores impulsan la decisión. Evaluado con Accuracy, F1 ponderado y AUC-ROC.
- **Dijkstra ajustado por RF:** el algoritmo encuentra el camino más barato en la red de estados. El costo de transporte de una ruta se "subsidia" (reduce) en el grafo proporcionalmente a la probabilidad de venta predicha por el bosque, guiando al productor hacia mercados rentables.

**Limitaciones:**

- Costos de flete son proxies; validar con cotizaciones reales para uso operativo.
- Oferta/demanda en Vogel son proporcionales a frecuencia; datos de volumen real mejorarían el modelo.
- SARIMA no captura choques externos (clima, precios internacionales); podrían incorporarse como variables exógenas (SARIMAX).
- El Random Forest de PySpark MLlib no genera visualización de árbol individual; para interpretabilidad adicional se recomienda SHAP con la API pandas.

In [ ]:
spark.stop()
print("SparkSession detenida.")